# Deployment Impact Analysis

Given an `entity_guid`, a `timestamp`, and an `account_id`, this notebook:
1. Fetches all deployments in the 30 days before `timestamp` via NRQL on `ChangeTrackingEvent`
2. Stores deployment metadata in a list of JSON objects
3. Fetches entity details + related entities (1 hop) with golden metrics info
4. For each deployment — fetches alert violations on the root entity, related-entity violations, and golden-metric anomalies; stores everything in the deployment JSON

In [228]:
import asyncio
import json
import statistics
import time
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple, Union

import httpx

In [229]:
# ── Configuration ────────────────────────────────────────────────────────────
API_KEY = "NRAK-YOUR-KEY-HERE"
NERD_GRAPH_HOST = "https://nerd-graph.staging-service.nr-ops.net"
REQUEST_TIMEOUT = 120

# ── Inputs ───────────────────────────────────────────────────────────────────
ENTITY_GUID = "MTA4ODg2MjN8QVBNfEFQUExJQ0FUSU9OfDMxNjg4ODA3OA"   # <-- change me
TIMESTAMP_MS = 1781196340360                     # <-- epoch ms; change me
ACCOUNT_ID = 10888623                             # <-- change me                          # <-- change me

# 30 days before the timestamp
THIRTY_DAYS_MS = 30 * 24 * 60 * 60 * 1000
WINDOW_AFTER_DEPLOY_MS = 2 * 60 * 60 * 1000     # how far after each deploy we look for violations / anomalies

## GraphQL / NRQL query strings

In [230]:
# ── Run a raw NRQL query ──────────────────────────────────────────────────────
NRQL_NG_QUERY = """
query NRQLQuery($accountId: Int!, $nrqlQuery: Nrql!, $timeout: Seconds) {
  actor {
    account(id: $accountId) {
      nrql(query: $nrqlQuery, timeout: $timeout) {
        results
        nrql
      }
    }
  }
}
"""

# ── Alert violations for an entity in a time window ──────────────────────────
GET_ENTITY_ISSUES_QUERY = """
query GetEntityIssues($guid: EntityGuid!, $startTime: EpochMilliseconds!, $endTime: EpochMilliseconds!) {
  actor {
    entity(guid: $guid) {
      guid
      name
      domain
      entityType
      alertSeverity
      alertViolations(startTime: $startTime, endTime: $endTime) {
        closedAt
        label
        level
        openedAt
      }
    }
  }
}
"""

# ── Entity details + 1-hop related entities with golden metrics ───────────────
ENTITY_AND_RELATIONSHIPS_QUERY = """
query EntityAndRelationships(
  $entityGuid: EntityGuid!,
  $hops: Int,
  $limit: Int,
  $hopFilters: [EntityRelationshipHopFilter!],
  $cursor: String
) {
  actor {
    entity(guid: $entityGuid) {
      accountId
      domain
      type
      name
      alertSeverity
      permalink
      guid
      goldenMetrics {
        metrics {
          name
          metricName
          definition {
            select
            from
            where
            facet
          }
        }
      }
      tags { key values }
      relationshipTraversal(
        hops: $hops
        limit: $limit
        hopFilters: { hopFilters: $hopFilters }
        cursor: $cursor
      ) {
        nextCursor
        results {
          type
          source {
            guid
            entity {
              guid
              name
              alertSeverity
              accountId
              domain
              type
              entityType
              reporting
              goldenMetrics {
                metrics {
                  name
                  metricName
                  definition { select from where facet }
                }
              }
              account { id name }
            }
          }
          target {
            guid
            entity {
              guid
              name
              alertSeverity
              accountId
              domain
              type
              entityType
              reporting
              goldenMetrics {
                metrics {
                  name
                  metricName
                  definition { select from where facet }
                }
              }
              account { id name }
            }
          }
        }
      }
    }
  }
}
"""

## Core helper functions

In [231]:
def get_request_headers() -> Dict[str, str]:
    return {
        "Api-Key": API_KEY,
        "Content-Type": "application/json",
        "NewRelic-Requesting-Services": "testing",
        "X-Login-Context": "",
        "X-Query-Source-Capability-Id": "NRAI",
    }


async def query_nerdgraph(
    query: str,
    variables: Optional[Dict] = None,
    operation_name: Optional[str] = None,
) -> httpx.Response:
    """Execute a NerdGraph GraphQL request and return the raw httpx.Response."""
    headers = get_request_headers()
    url = f"{NERD_GRAPH_HOST}/graphql"
    async with httpx.AsyncClient() as client:
        body: Dict[str, Any] = {"query": query}
        if variables is not None:
            body["variables"] = variables
        if operation_name is not None:
            body["operationName"] = operation_name
        t0 = time.perf_counter()
        resp = await client.post(url, headers=headers, json=body, timeout=REQUEST_TIMEOUT)
        print(f"NerdGraph: {resp.status_code} in {time.perf_counter() - t0:.2f}s")
        resp.raise_for_status()
    return resp


def _get_nested_value(data_dict: dict, dot_key: str) -> Any:
    """Navigate a nested dict with dot-notation (e.g. 'data.actor.account.nrql.results')."""
    current = data_dict
    try:
        for key in dot_key.split("."):
            current = current[key]
        return current
    except (KeyError, TypeError):
        return None


async def query_nrql(nrql: str, account_id: int, timeout: int = 60) -> List[dict]:
    """Run a NRQL query via NerdGraph and return the results list."""
    var = {"nrqlQuery": nrql, "accountId": account_id, "timeout": timeout}
    try:
        resp = await query_nerdgraph(NRQL_NG_QUERY, var)
        parsed = json.loads(resp.text)
        result = _get_nested_value(parsed, "data.actor.account.nrql.results")
        if result is None:
            print(f"[query_nrql] Could not parse response: {parsed}")
            return []
        return result
    except Exception as e:
        print(f"[query_nrql] Error running '{nrql[:80]}...': {e}")
        return []


def ms_to_date_string(timestamp_ms: int) -> str:
    """Convert epoch-milliseconds to a human-readable string."""
    try:
        return datetime.fromtimestamp(timestamp_ms / 1000).strftime("%Y-%m-%d %H:%M:%S")
    except Exception as e:
        return f"Error: {e}"

## Step 1+2 — Fetch deployments (30 days before timestamp) + store metadata

In [232]:
def build_change_events_nrql(entity_guid: str, since_ms: int, until_ms: int) -> str:
    """
    Builds the ChangeTrackingEvent NRQL query used in hackathon.ipynb
    (nrql_query_for_change_events), parameterised by entity guid and time window.
    """
    return f"""
SELECT
  changeTrackingId,
  timestamp,
  deployment_result,
  deployment_phase,
  version,
  `entity.guid`,
  `entity.name`,
  gheOrg,
  gheRepo,
  team,
  environment,
  commit,
  changelog,
  type,
  user,
  groupId,
  configurationVersion,
  deployMechanism
FROM ChangeTrackingEvent
WHERE category = 'Deployment'
  AND deployment_phase = 'end'
  AND `entity.guid` = '{entity_guid}'
SINCE {since_ms} UNTIL {until_ms}
LIMIT MAX
"""


async def fetch_deployments(
    entity_guid: str,
    timestamp_ms: int,
    account_id: int,
    lookback_ms: int = THIRTY_DAYS_MS,
) -> List[Dict]:
    """
    Step 1+2: Fetch all deployments (ChangeTrackingEvent) for the entity
    in the 30-day window before `timestamp_ms`.

    Returns a list of deployment metadata dicts, each ready to be enriched
    with alerts / anomalies in Step 4.
    """
    since_ms = timestamp_ms - lookback_ms
    nrql = build_change_events_nrql(entity_guid, since_ms, timestamp_ms)
    print(f"Fetching deployments from {ms_to_date_string(since_ms)} to {ms_to_date_string(timestamp_ms)} ...")
    rows = await query_nrql(nrql, account_id)
    deployments = []
    for row in rows:
        deployments.append({
            # core metadata
            "changeTrackingId": row.get("changeTrackingId"),
            "timestamp_ms": row.get("timestamp"),
            "timestamp_str": ms_to_date_string(row.get("timestamp", 0)),
            "entity_guid": row.get("entity.guid"),
            "entity_name": row.get("entity.name"),
            "version": row.get("version"),
            "deployment_result": row.get("deployment_result"),
            "user": row.get("user"),
            "commit": row.get("commit"),
            "changelog": row.get("changelog"),
            "team": row.get("team"),
            "environment": row.get("environment"),
            "gheOrg": row.get("gheOrg"),
            "gheRepo": row.get("gheRepo"),
            "groupId": row.get("groupId"),
            "type": row.get("type"),
            "deployMechanism": row.get("deployMechanism"),
            "configurationVersion": row.get("configurationVersion"),
            # placeholders for Step 4
            "root_entity_alert_violations": [],
            "root_entity_anomalies": {},
            "related_entities": [],
        })
    print(f"Found {len(deployments)} deployment(s).")
    return deployments

## Step 3 — Fetch entity details + related entities (1 hop)

In [233]:
# TODO - Filter on just entity type = "APM_APPLICATION"

SUPPORTED_RELATIONSHIP_TYPES = {
    "CALLS", "CONNECTS_TO", "CONSUMES", "CONTAINS", "HOSTS", "PRODUCES", "SERVES",
}

EXCLUDED_DOMAIN_TYPES = [
    {"domain": "AIOPS",  "type": "CONDITION"},
    {"domain": "AIOPS",  "type": "ISSUE"},
    {"domain": "REF",    "type": "REPOSITORY"},
    {"domain": "SYNTH",  "type": "SECURE_CRED"},
    {"domain": "SYNTH",  "type": "PRIVATE_LOCATION"},
    {"domain": "VIZ",    "type": "DASHBOARD"},
]


async def fetch_entity_and_related(
    entity_guid: str,
    hops: int = 1,
    limit: int = 50,
) -> Tuple[Dict, Dict[str, Dict]]:
    """
    Step 3: Fetch root entity details and all 1-hop related entities.

    Returns:
        root_entity  : dict with keys guid, name, accountId, domain, type,
                       alertSeverity, permalink, goldenMetrics, tags
        related_map  : {guid -> entity_dict} for every connected 1-hop entity
                       (filtered to SUPPORTED_RELATIONSHIP_TYPES, ISSUE type excluded)
    """
    hop_filter = {
        "hop": hops,
        "filters": [{
            "direction": "BOTH",
            "toEntityDomainTypes":   {"exclude": EXCLUDED_DOMAIN_TYPES},
            "fromEntityDomainTypes": {"exclude": EXCLUDED_DOMAIN_TYPES},
        }],
    }
    variables = {
        "entityGuid": entity_guid,
        "hops": hops,
        "limit": limit,
        "hopFilters": [hop_filter],
        "cursor": None,
    }
    resp = await query_nerdgraph(ENTITY_AND_RELATIONSHIPS_QUERY, variables)
    data = resp.json()["data"]["actor"]["entity"]

    root_entity = {
        "guid":         data["guid"],
        "name":         data["name"],
        "accountId":    data["accountId"],
        "domain":       data["domain"],
        "type":         data["type"],
        "alertSeverity": data.get("alertSeverity"),
        "permalink":    data.get("permalink"),
        "goldenMetrics": data.get("goldenMetrics"),
        "tags":         data.get("tags"),
    }

    related_map: Dict[str, Dict] = {}
    for rel in data["relationshipTraversal"]["results"]:
        if rel["type"] not in SUPPORTED_RELATIONSHIP_TYPES:
            continue
        for side in ("source", "target"):
            ent = rel[side]["entity"]
            guid = ent["guid"]
            if guid != entity_guid and ent.get("type") != "ISSUE" and guid not in related_map:
                related_map[guid] = ent

    print(f"Root entity: {root_entity['name']} ({root_entity['guid']})")
    print(f"Related entities (1 hop): {len(related_map)}")
    return root_entity, related_map

## Alert violations helper

In [234]:
async def fetch_alert_violations(
    entity_guid: str,
    start_time_ms: int,
    end_time_ms: int,
) -> Dict:
    """
    Fetch alert violations for a single entity in [start_time_ms, end_time_ms].

    Returns a dict with keys: guid, name, domain, entityType, alertSeverity,
    alertViolations (list).  Returns an empty dict on error.
    """
    try:
        resp = await query_nerdgraph(
            GET_ENTITY_ISSUES_QUERY,
            {"guid": entity_guid, "startTime": start_time_ms, "endTime": end_time_ms},
        )
        entity_data = resp.json().get("data", {}).get("actor", {}).get("entity") or {}
        # normalise timestamps in violations to human-readable strings
        for v in entity_data.get("alertViolations", []):
            if v.get("openedAt"):
                v["openedAt"] = ms_to_date_string(v["openedAt"])
            if v.get("closedAt"):
                v["closedAt"] = ms_to_date_string(v["closedAt"])
        return entity_data
    except Exception as e:
        print(f"[fetch_alert_violations] {entity_guid}: {e}")
        return {}


def has_violations(entity_data: Dict) -> bool:
    """Return True if the entity has at least one alert violation AND a WARNING/CRITICAL severity."""
    return (
        len(entity_data.get("alertViolations", [])) > 0
        and entity_data.get("alertSeverity") in ("WARNING", "CRITICAL")
    )

## Anomaly detection on golden metrics

In [235]:
async def three_sigma(
    signal: List[Tuple[int, float]],
    train_end_index: int,
) -> Tuple[List[int], str, float]:
    """
    Three-sigma anomaly detector (ported from rca.py).

    Args:
        signal          : list of (timestamp_seconds, value) tuples
        train_end_index : index that splits training vs test data

    Returns:
        (anomaly_indices, direction, avg_score)
        direction is 'Increased', 'Decreased', or 'unknown'
    """
    direction = "unknown"
    if len(signal) < train_end_index or len(signal) == 0:
        return [], direction, 0.0

    # strip invalid values
    valid = [v for _, v in signal if isinstance(v, (int, float))]
    if not valid:
        return [], direction, 0.0
    if len(valid) < len(signal):
        mean_val = statistics.mean(valid)
        signal = [(t, v) if isinstance(v, (int, float)) else (t, mean_val) for t, v in signal]

    train_end_index = abs(train_end_index - 5)
    training = signal[:train_end_index]
    test     = signal[train_end_index:]

    if len(training) < 2:
        return [], direction, 0.0

    mean    = statistics.mean([v for _, v in training])
    std_dev = statistics.stdev([v for _, v in training])

    anomalies: List[int] = []
    delta = 0
    for idx, (_, dp) in enumerate(test):
        if dp < (mean - 3 * std_dev):
            delta -= 1
            anomalies.append(idx + train_end_index)
        elif dp > (mean + 3 * std_dev):
            delta += 1
            anomalies.append(idx + train_end_index)

    if delta > 0:
        direction = "Increased"
    elif delta < 0:
        direction = "Decreased"

    return anomalies, direction, 0.0


async def find_anomalies_for_golden_metric(
    gm_name: str,
    gm_definition: Dict,          # {"select": .., "from": .., "where": .., "facet": ..}
    entity_guid: str,
    start_time_ms: int,
    end_time_ms: int,
    account_id: int,
) -> Tuple[str, Optional[List], str, List]:
    """
    Build and run the golden-metric NRQL query, then run three-sigma anomaly detection.

    `start_time_ms` should be BEFORE the deployment so the detector has training data.

    Returns (metric_name, anomaly_timestamps_or_None, direction, raw_gm_data)
    """
    gm_data: List = []
    try:
        # add a 75-minute training buffer before start_time so three_sigma has baseline data
        train_buffer_ms  = 75 * 60 * 1000
        adjusted_start   = int(start_time_ms - train_buffer_ms)
        gm_name_cleaned  = gm_name.replace(".", "_")

        where_clause = gm_definition.get("where")
        if where_clause:
            nrql = (
                f"FROM {gm_definition['from']} "
                f"SELECT {gm_definition['select']} AS {gm_name_cleaned} "
                f"WHERE {where_clause} AND entity.guid = '{entity_guid}' "
                f"SINCE {adjusted_start} UNTIL {end_time_ms} TIMESERIES"
            )
        else:
            nrql = (
                f"FROM {gm_definition['from']} "
                f"SELECT {gm_definition['select']} AS {gm_name_cleaned} "
                f"WHERE entity.guid = '{entity_guid}' "
                f"SINCE {adjusted_start} UNTIL {end_time_ms} TIMESERIES"
            )

        rows = await query_nrql(nrql, account_id)
        for row in rows:
            dp = row.get(gm_name_cleaned)
            if isinstance(dp, dict):
                dp = next(iter(dp.values()))
            gm_data.append((row.get("beginTimeSeconds", 0), dp))

        # train on the buffer portion
        total_duration_ms = end_time_ms - adjusted_start
        train_end_index   = int((len(gm_data) * train_buffer_ms) / total_duration_ms) if total_duration_ms else 0

        anomaly_indices, direction, _ = await three_sigma(gm_data, train_end_index)
        anomaly_timestamps = [gm_data[i][0] for i in anomaly_indices]
        return gm_name, anomaly_timestamps, direction, gm_data
    except Exception as e:
        print(f"[find_anomalies_for_golden_metric] {gm_name}: {e}")
        return gm_name, None, "unknown", gm_data


async def get_golden_metric_anomalies(
    entity_data: Dict,            # entity dict returned by fetch_entity_and_related
    start_time_ms: int,
    end_time_ms: int,
) -> Dict:
    """
    For a given entity dict (with goldenMetrics), detect anomalies in every metric.

    Returns a dict keyed by metric_name with anomaly info; empty dict if none found.
    """
    golden_metrics = (entity_data.get("goldenMetrics") or {}).get("metrics") or []
    account_id     = entity_data.get("accountId") or (entity_data.get("account") or {}).get("id")
    guid           = entity_data["guid"]

    if not golden_metrics or not account_id:
        return {}

    tasks = [
        find_anomalies_for_golden_metric(
            gm_name     = m["metricName"],
            gm_definition = m["definition"],
            entity_guid = guid,
            start_time_ms = start_time_ms,
            end_time_ms   = end_time_ms,
            account_id    = account_id,
        )
        for m in golden_metrics
        if m and m.get("definition")
    ]

    results = await asyncio.gather(*tasks)
    anomalies_found: Dict = {}
    for name, timestamps, direction, _ in results:
        if timestamps:
            anomalies_found[name] = {
                "anomaly_count": len(timestamps),
                "first_anomaly_at": ms_to_date_string(min(timestamps) * 1000),
                "direction": direction,
            }
    return anomalies_found

## Step 4 — Enrich each deployment with violations + anomalies

In [236]:
async def enrich_deployment(
    deployment: Dict,
    root_entity: Dict,
    related_map: Dict[str, Dict],
    account_id: int,
    window_after_ms: int = WINDOW_AFTER_DEPLOY_MS,
) -> Dict:
    """
    For a single deployment, fetch alert violations and golden-metric anomalies
    for the root entity and its 1-hop related entities, following the logic:

    1. Fetch alert violations for root entity.
    2. If root has violations:
       a. Fetch violations for all related entities.
       b. If NO related entities have violations → detect anomalies in related entities' GMs.
    3. If root has NO violations:
       a. Detect anomalies in root entity GMs.
       b. If anomalies found → fetch violations + anomalies for related entities.

    All results are stored inside the deployment dict (in-place + returned).
    """
    deploy_ts_ms = deployment.get("timestamp_ms") or 0
    start_ms     = deploy_ts_ms
    end_ms       = deploy_ts_ms + window_after_ms

    print(f"\n── Deployment {deployment.get('changeTrackingId')} @ {deployment.get('timestamp_str')} ──")

    # ── 1. Root entity alert violations ──────────────────────────────────────
    root_violation_data = await fetch_alert_violations(root_entity["guid"], start_ms, end_ms)
    root_violations     = root_violation_data.get("alertViolations", [])
    deployment["root_entity_alert_violations"] = root_violations
    print(f"  Root violations: {len(root_violations)}")

    related_results: List[Dict] = []

    if has_violations(root_violation_data):
        # ── 2a. Fetch violations for related entities ─────────────────────────
        print(f"  Root has violations → checking {len(related_map)} related entities ...")
        related_violation_tasks = [
            fetch_alert_violations(guid, start_ms, end_ms)
            for guid in related_map
        ]
        related_violation_results = await asyncio.gather(*related_violation_tasks)

        any_related_violation = False
        for guid, vdata in zip(related_map.keys(), related_violation_results):
            rel_info = dict(related_map[guid])   # copy
            rel_info["alert_violations"] = vdata.get("alertViolations", [])
            rel_info["alertSeverity"]    = vdata.get("alertSeverity")
            rel_info["anomalies"]        = {}

            if has_violations(vdata):
                any_related_violation = True
            related_results.append(rel_info)

        # ── 2b. No related violations → check anomalies in related GMs ───────
        if not any_related_violation:
            print("  No related violations → running GM anomaly detection on related entities ...")
            anomaly_tasks = [
                get_golden_metric_anomalies(rel_info, start_ms, end_ms)
                for rel_info in related_results
            ]
            anomaly_results = await asyncio.gather(*anomaly_tasks)
            for rel_info, anomalies in zip(related_results, anomaly_results):
                rel_info["anomalies"] = anomalies

    else:
        # ── 3a. Root has no violations → check root GM anomalies ─────────────
        print("  Root has no violations → running GM anomaly detection on root entity ...")
        root_anomalies = await get_golden_metric_anomalies(root_entity, start_ms, end_ms)
        deployment["root_entity_anomalies"] = root_anomalies
        print(f"  Root anomalies: {len(root_anomalies)} metric(s) with anomalies")

        if root_anomalies:
            # ── 3b. Root anomalies → track violations + anomalies in related ─
            print(f"  Root anomalies found → checking {len(related_map)} related entities ...")
            for guid, ent in related_map.items():
                rel_info = dict(ent)
                vdata    = await fetch_alert_violations(guid, start_ms, end_ms)
                rel_info["alert_violations"] = vdata.get("alertViolations", [])
                rel_info["alertSeverity"]    = vdata.get("alertSeverity")
                rel_info["anomalies"]        = await get_golden_metric_anomalies(ent, start_ms, end_ms)
                related_results.append(rel_info)
        else:
            # No anomalies anywhere — just record empty related-entity stubs
            for guid, ent in related_map.items():
                rel_info = dict(ent)
                rel_info["alert_violations"] = []
                rel_info["alertSeverity"]    = ent.get("alertSeverity")
                rel_info["anomalies"]        = {}
                related_results.append(rel_info)

    deployment["related_entities"] = related_results
    return deployment

## Main orchestrator

In [237]:
async def run_deployment_impact_analysis(
    entity_guid: str,
    timestamp_ms: int,
    account_id: int,
) -> List[Dict]:
    """
    Full pipeline:
      1+2. Fetch deployments (30 days before timestamp_ms)
      3.   Fetch entity details + related entities
      4.   Enrich each deployment with violations / anomalies

    Returns the enriched list of deployment dicts.
    """
    # Steps 1+2
    deployments = await fetch_deployments(entity_guid, timestamp_ms, account_id)
    if not deployments:
        print("No deployments found in the last 30 days.")
        return []

    # Step 3
    root_entity, related_map = await fetch_entity_and_related(entity_guid, hops=1)

    # Step 4
    print(f"\nEnriching {len(deployments)} deployment(s) ...")
    enriched = []
    for deployment in deployments:
        enriched_dep = await enrich_deployment(
            deployment  = deployment,
            root_entity = root_entity,
            related_map = related_map,
            account_id  = account_id,
        )
        enriched.append(enriched_dep)

    return enriched

## Run the analysis

In [ ]:
# results = await run_deployment_impact_analysis(
#     entity_guid  = ENTITY_GUID,
#     timestamp_ms = TIMESTAMP_MS,
#     account_id   = ACCOUNT_ID,
# )

# print(f"\n=== Done. {len(results)} deployment(s) enriched. ===")

Fetching deployments from 2026-05-12 22:15:40 to 2026-06-11 22:15:40 ...
NerdGraph: 200 in 1.44s
Found 17 deployment(s).
NerdGraph: 200 in 1.24s
Root entity: Condition Standards Auditor (production) (MTA4ODg2MjN8QVBNfEFQUExJQ0FUSU9OfDMxNjg4ODA3OA)
Related entities (1 hop): 0

Enriching 17 deployment(s) ...

── Deployment 8b0a508e-088b-4aa2-b434-9708ebc2cd7a @ 2026-05-13 21:06:03 ──
NerdGraph: 200 in 1.22s
  Root violations: 1
  Root has no violations → running GM anomaly detection on root entity ...
NerdGraph: 200 in 1.23s
NerdGraph: 200 in 1.33s
NerdGraph: 200 in 1.32s
  Root anomalies: 0 metric(s) with anomalies

── Deployment 0d4a8419-ab57-42c2-8498-ba451c31d604 @ 2026-05-13 04:28:30 ──
NerdGraph: 200 in 1.19s
  Root violations: 1
  Root has no violations → running GM anomaly detection on root entity ...
NerdGraph: 200 in 1.58s
NerdGraph: 200 in 1.59s
NerdGraph: 200 in 1.59s
  Root anomalies: 0 metric(s) with anomalies

── Deployment 0d4a8419-ab57-42c2-8498-ba451c31d604 @ 2026-05-13

## Step 4 (Parallel) — Enrich deployments concurrently

Runs enrichment for all deployments in parallel using `asyncio.gather` instead of sequentially. This is significantly faster when there are many deployments.

In [ ]:
async def run_deployment_impact_analysis_parallel(
    entity_guid: str,
    timestamp_ms: int,
    account_id: int,
    max_concurrent: int = 5,
) -> List[Dict]:
    """
    Same as run_deployment_impact_analysis but enriches deployments in parallel.
    
    Uses asyncio.Semaphore to limit concurrency and avoid overwhelming the API.
    
    Args:
        max_concurrent: Maximum number of deployments enriched simultaneously (default 5)
    """
    # Steps 1+2: Fetch deployments
    deployments = await fetch_deployments(entity_guid, timestamp_ms, account_id)
    if not deployments:
        print("No deployments found in the last 30 days.")
        return []

    # Step 3: Fetch entity details + related entities
    root_entity, related_map = await fetch_entity_and_related(entity_guid, hops=1)

    # Step 4: Enrich ALL deployments in parallel (with concurrency limit)
    print(f"\nEnriching {len(deployments)} deployment(s) in parallel (max {max_concurrent} concurrent)...")
    
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def enrich_with_limit(deployment):
        async with semaphore:
            return await enrich_deployment(
                deployment=deployment,
                root_entity=root_entity,
                related_map=related_map,
                account_id=account_id,
            )
    
    t0 = time.perf_counter()
    enriched = await asyncio.gather(*[enrich_with_limit(d) for d in deployments])
    elapsed = time.perf_counter() - t0
    
    print(f"\n✅ All {len(enriched)} deployments enriched in {elapsed:.1f}s (parallel)")
    print(f"   vs sequential estimate: ~{elapsed * max_concurrent:.0f}s")
    
    return list(enriched)


# Run the parallel version
results = await run_deployment_impact_analysis_parallel(
    entity_guid=ENTITY_GUID,
    timestamp_ms=TIMESTAMP_MS,
    account_id=ACCOUNT_ID,
    max_concurrent=5,  # Adjust if API rate limits
)

print(f"\n=== Done. {len(results)} deployment(s) enriched. ===")

In [116]:
# Inspect results
for dep in results:
    print(f"\nDeployment: {dep.get('changeTrackingId')} @ {dep.get('timestamp_str')}")
    print(f"  version              : {dep.get('version')}")
    print(f"  user                 : {dep.get('user')}")
    print(f"  root violations      : {len(dep.get('root_entity_alert_violations', []))}")
    print(f"  root anomalies       : {list(dep.get('root_entity_anomalies', {}).keys())}")
    for rel in dep.get("related_entities", []):
        n_viol = len(rel.get("alert_violations", []))
        n_anom = len(rel.get("anomalies", {}))
        if n_viol or n_anom:
            print(f"  ↳ {rel.get('name')} ({rel.get('guid')}) — violations={n_viol}, anomalies={n_anom}")


Deployment: 1c8ac57f-a630-475c-8f9b-1f7cb315707f @ 2026-06-10 08:34:01
  version              : release-408
  user                 : mlaspina
  root violations      : 0
  root anomalies       : []

Deployment: c0075c28-29a1-47c9-bc26-81c872954249 @ 2026-06-05 07:02:31
  version              : release-407
  user                 : mlaspina
  root violations      : 3
  root anomalies       : []

Deployment: b6fccdc3-c81e-4dfe-9be7-660f26cbf0b9 @ 2026-05-21 06:49:32
  version              : release-405
  user                 : mlaspina
  root violations      : 2
  root anomalies       : []

Deployment: 8cb09ac7-8f38-4380-9875-5721ec99a06e @ 2026-05-15 05:55:16
  version              : release-403
  user                 : mlaspina
  root violations      : 1
  root anomalies       : []


In [117]:
# Pretty-print full JSON for the first deployment (or all of them)
if results:
    print(json.dumps(results[0], indent=2, default=str))

{
  "changeTrackingId": "1c8ac57f-a630-475c-8f9b-1f7cb315707f",
  "timestamp_ms": 1781060641727,
  "timestamp_str": "2026-06-10 08:34:01",
  "entity_guid": "Nzc5ODIwfEFQTXxBUFBMSUNBVElPTnwxNzQzMjgyNTI",
  "entity_name": "query-summarizer (production.us-sad-sandwich)",
  "version": "release-408",
  "deployment_result": "success",
  "user": "mlaspina",
  "commit": "7550f2dc9830ac290659443dfb776cc5e78e2719",
  "changelog": "https://source.datanerd.us/derived-data-streams/query-summarizer/releases/tag/release-408",
  "team": "Spyglass",
  "environment": "us-sad-sandwich",
  "gheOrg": "derived-data-streams",
  "gheRepo": "query-summarizer",
  "groupId": "1673-query-summarizer-us-sad-sandwich-8649447",
  "type": "Rolling",
  "deployMechanism": "kubernetes",
  "configurationVersion": "release-408",
  "root_entity_alert_violations": [],
  "root_entity_anomalies": {},
  "related_entities": [
    {
      "account": {
        "id": 1,
        "name": "NewRelic Administration"
      },
      "acco

In [ ]:
# TODO - change logs
# TODO  - Prompt - dump all data, guidelines for risk score, actions depending on risk score.

## Step 5 — Fetch Changelogs and Code Diffs from GHE

For a given deployment, uses the `changelog` URL and `commit` SHA from Change Tracking to fetch the actual code diff from GitHub Enterprise.

In [171]:
import re
import os

# ── Configuration ────────────────────────────────────────────────────────────
GHE_TOKEN = os.environ.get("GHE_TOKEN", "")  # Set your GHE token here or via env var
GHE_API_BASE = "https://source.datanerd.us/api/v3"


async def ghe_get(path: str) -> Optional[Dict]:
    """GET request to GitHub Enterprise API."""
    if not GHE_TOKEN:
        print("[ghe_get] GHE_TOKEN not set — skipping GHE API call")
        return None
    url = f"{GHE_API_BASE}{path}"
    headers = {
        "Authorization": f"token {GHE_TOKEN}",
        "Accept": "application/vnd.github.v3+json",
    }
    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(url, headers=headers, timeout=15)
            resp.raise_for_status()
            return resp.json()
    except httpx.HTTPStatusError as e:
        print(f"[ghe_get] HTTP {e.response.status_code}: {path}")
        return None
    except Exception as e:
        print(f"[ghe_get] Error: {e}")
        return None


def parse_changelog_url(changelog_url: str) -> Optional[Dict[str, str]]:
    """
    Parse a changelog URL from Change Tracking into org/repo/tag.
    
    Input:  "https://source.datanerd.us/Alerting/condition-api/releases/tag/release-1174"
    Output: {"org": "Alerting", "repo": "condition-api", "tag": "release-1174"}
    """
    if not changelog_url:
        return None
    match = re.match(
        r"https://source\.datanerd\.us/([^/]+)/([^/]+)/releases/tag/(.+)",
        changelog_url,
    )
    if match:
        return {"org": match.group(1), "repo": match.group(2), "tag": match.group(3)}
    # Also handle PR links
    match = re.match(
        r"https://source\.datanerd\.us/([^/]+)/([^/]+)/pull/(\d+)",
        changelog_url,
    )
    if match:
        return {"org": match.group(1), "repo": match.group(2), "pr": match.group(3)}
    return None


async def fetch_code_diff_from_changelog(changelog_url: str) -> Dict:
    """
    Given a changelog URL, fetch the code diff by comparing with the previous tag.
    
    Steps:
      1. Parse URL → org/repo/tag
      2. GET /repos/{org}/{repo}/tags → find previous tag
      3. GET /repos/{org}/{repo}/compare/{prev_tag}...{current_tag} → full diff
    
    Returns:
      {
        "org": "Alerting",
        "repo": "condition-api",
        "tag": "release-1174",
        "previous_tag": "release-1171",
        "total_commits": 5,
        "files_changed": 12,
        "lines_added": 347,
        "lines_removed": 89,
        "files": [
          {"filename": "src/Handler.java", "additions": 45, "deletions": 12, "status": "modified"},
          ...
        ]
      }
    """
    parsed = parse_changelog_url(changelog_url)
    if not parsed:
        return {"error": f"Cannot parse URL: {changelog_url}"}
    
    org = parsed["org"]
    repo = parsed["repo"]
    
    # Handle PR links differently
    if "pr" in parsed:
        pr_data = await ghe_get(f"/repos/{org}/{repo}/pulls/{parsed['pr']}/files")
        if not pr_data:
            return {"error": "Could not fetch PR files", "org": org, "repo": repo}
        return {
            "org": org,
            "repo": repo,
            "pr_number": parsed["pr"],
            "files_changed": len(pr_data),
            "lines_added": sum(f.get("additions", 0) for f in pr_data),
            "lines_removed": sum(f.get("deletions", 0) for f in pr_data),
            "files": [
                {"filename": f["filename"], "additions": f.get("additions", 0),
                 "deletions": f.get("deletions", 0), "status": f.get("status", "")}
                for f in pr_data
            ],
        }
    
    tag = parsed["tag"]
    
    # Step 2: Get tags to find previous
    tags_data = await ghe_get(f"/repos/{org}/{repo}/tags?per_page=30")
    if not tags_data:
        return {"error": "Could not fetch tags", "org": org, "repo": repo, "tag": tag}
    
    tag_names = [t["name"] for t in tags_data]
    try:
        idx = tag_names.index(tag)
        prev_tag = tag_names[idx + 1] if idx + 1 < len(tag_names) else None
    except ValueError:
        prev_tag = None
    
    if not prev_tag:
        return {"error": "Could not find previous tag", "org": org, "repo": repo, "tag": tag}
    
    # Step 3: Compare tags
    compare_data = await ghe_get(f"/repos/{org}/{repo}/compare/{prev_tag}...{tag}")
    if not compare_data:
        return {"error": "Compare failed", "org": org, "repo": repo, "tag": tag, "previous_tag": prev_tag}
    
    files = compare_data.get("files", [])
    return {
        "org": org,
        "repo": repo,
        "tag": tag,
        "previous_tag": prev_tag,
        "total_commits": len(compare_data.get("commits", [])),
        "files_changed": len(files),
        "lines_added": sum(f.get("additions", 0) for f in files),
        "lines_removed": sum(f.get("deletions", 0) for f in files),
        "files": [
            {"filename": f["filename"], "additions": f.get("additions", 0),
             "deletions": f.get("deletions", 0), "status": f.get("status", "")}
            for f in files
        ],
    }


async def fetch_code_diff_from_commit(org: str, repo: str, commit_sha: str) -> Dict:
    """
    Fetch diff for a single commit SHA.
    
    Returns:
      {
        "commit": "0dcc14a30cf8",
        "message": "Refactor condition evaluation",
        "author": "mdiener",
        "files_changed": 8,
        "lines_added": 234,
        "lines_removed": 56,
        "files": [...]
      }
    """
    commit_data = await ghe_get(f"/repos/{org}/{repo}/commits/{commit_sha}")
    if not commit_data:
        return {"error": "Could not fetch commit", "org": org, "repo": repo, "commit": commit_sha}
    
    stats = commit_data.get("stats", {})
    files = commit_data.get("files", [])
    commit_info = commit_data.get("commit", {})
    
    return {
        "org": org,
        "repo": repo,
        "commit": commit_sha[:12],
        "message": (commit_info.get("message") or "").split("\n")[0],
        "author": (commit_info.get("author") or {}).get("name", ""),
        "files_changed": len(files),
        "lines_added": stats.get("additions", 0),
        "lines_removed": stats.get("deletions", 0),
        "files": [
            {"filename": f["filename"], "additions": f.get("additions", 0),
             "deletions": f.get("deletions", 0), "status": f.get("status", "")}
            for f in files
        ],
    }


async def fetch_code_diff_for_deployment(deployment: Dict) -> Dict:
    """
    Given a deployment dict (from Step 1+2), fetch its code diff.
    
    Tries changelog URL first (tag comparison), falls back to commit SHA.
    """
    changelog = deployment.get("changelog", "")
    commit = deployment.get("commit", "")
    ghe_org = deployment.get("gheOrg", "")
    ghe_repo = deployment.get("gheRepo", "")
    
    # Try changelog URL first (gives full release diff)
    if changelog:
        print(f"  Fetching diff from changelog: {changelog}")
        diff = await fetch_code_diff_from_changelog(changelog)
        if "error" not in diff:
            return diff
        print(f"  Changelog failed: {diff.get('error')} — falling back to commit")
    
    # Fallback: single commit diff
    if commit and ghe_org and ghe_repo:
        print(f"  Fetching diff from commit: {ghe_org}/{ghe_repo}@{commit[:12]}")
        return await fetch_code_diff_from_commit(ghe_org, ghe_repo, commit)
    
    return {"error": "No changelog URL or commit SHA available"}


print("✅ Code diff functions defined. Set GHE_TOKEN to fetch real diffs.")

✅ Code diff functions defined. Set GHE_TOKEN to fetch real diffs.


In [172]:
# Fetch code diff for a specific deployment (pick one from results)
# Change the index to analyze a different deployment

TARGET_DEPLOYMENT_INDEX = 0  # <-- change this to pick a different deployment

if results:
    target_dep = results[TARGET_DEPLOYMENT_INDEX]
    print(f"Fetching code diff for deployment:")
    print(f"  Version:   {target_dep.get('version')}")
    print(f"  Commit:    {target_dep.get('commit', '')[:12]}")
    print(f"  Changelog: {target_dep.get('changelog', 'N/A')}")
    print(f"  Repo:      {target_dep.get('gheOrg')}/{target_dep.get('gheRepo')}")
    print()

    code_diff = await fetch_code_diff_for_deployment(target_dep)

    if "error" in code_diff:
        print(f"\n❌ Error: {code_diff['error']}")
    else:
        print(f"\n✅ Code diff fetched successfully:")
        print(f"  Tag:             {code_diff.get('tag', code_diff.get('commit', ''))}")
        print(f"  Previous tag:    {code_diff.get('previous_tag', 'N/A')}")
        print(f"  Commits:         {code_diff.get('total_commits', 'N/A')}")
        print(f"  Files changed:   {code_diff.get('files_changed', 0)}")
        print(f"  Lines added:     +{code_diff.get('lines_added', 0)}")
        print(f"  Lines removed:   -{code_diff.get('lines_removed', 0)}")
        print(f"\n  Files:")
        for f in code_diff.get("files", [])[:20]:
            print(f"    {f['status']:>10}  +{f['additions']:<4} -{f['deletions']:<4}  {f['filename']}")
        if len(code_diff.get("files", [])) > 20:
            print(f"    ... and {len(code_diff['files']) - 20} more files")

    # Store it in the deployment dict for later use
    target_dep["code_diff"] = code_diff
else:
    print("No results — run the analysis cell first.")

Fetching code diff for deployment:
  Version:   release-408
  Commit:    7550f2dc9830
  Changelog: https://source.datanerd.us/derived-data-streams/query-summarizer/releases/tag/release-408
  Repo:      derived-data-streams/query-summarizer

  Fetching diff from changelog: https://source.datanerd.us/derived-data-streams/query-summarizer/releases/tag/release-408

✅ Code diff fetched successfully:
  Tag:             release-408
  Previous tag:    release-407
  Commits:         3
  Files changed:   2
  Lines added:     +3
  Lines removed:   -3

  Files:
      modified  +2    -2     dependency_license_manifest.yml
      modified  +1    -1     gradle/libs.versions.toml


## Step 6 — Fetch Actual Code Patches (Line-Level Changes)

Fetches the full patch/diff content for each file in the deployment — showing exactly which lines were added/removed. This enables identifying specific code that could cause production issues.

In [225]:
async def fetch_code_patches_from_changelog(changelog_url: str) -> Dict:
    """
    Fetch the full patch content (actual code lines changed) for a release.
    
    Uses the same compare API but extracts the 'patch' field from each file,
    which contains the unified diff showing exact line changes.
    
    Returns:
      {
        "org": "Alerting",
        "repo": "condition-api",
        "tag": "release-1174",
        "previous_tag": "release-1171",
        "total_commits": 5,
        "files": [
          {
            "filename": "src/main/java/Handler.java",
            "status": "modified",
            "additions": 45,
            "deletions": 12,
            "patch": "@@ -23,7 +23,12 @@ public class Handler {\n-    old line\n+    new line\n ..."
          },
          ...
        ],
        "commit_messages": ["Fix null pointer in evaluator", "Add retry logic", ...]
      }
    """
    parsed = parse_changelog_url(changelog_url)
    if not parsed:
        return {"error": f"Cannot parse URL: {changelog_url}"}
    
    org = parsed["org"]
    repo = parsed["repo"]
    
    if "pr" in parsed:
        # For PRs, fetch files with patches
        pr_files = await ghe_get(f"/repos/{org}/{repo}/pulls/{parsed['pr']}/files")
        if not pr_files:
            return {"error": "Could not fetch PR files", "org": org, "repo": repo}
        return {
            "org": org,
            "repo": repo,
            "pr_number": parsed["pr"],
            "files": [
                {
                    "filename": f["filename"],
                    "status": f.get("status", ""),
                    "additions": f.get("additions", 0),
                    "deletions": f.get("deletions", 0),
                    "patch": f.get("patch", ""),  # The actual unified diff
                }
                for f in pr_files
            ],
        }
    
    tag = parsed["tag"]
    
    # Find previous tag
    tags_data = await ghe_get(f"/repos/{org}/{repo}/tags?per_page=30")
    if not tags_data:
        return {"error": "Could not fetch tags", "org": org, "repo": repo, "tag": tag}
    
    tag_names = [t["name"] for t in tags_data]
    try:
        idx = tag_names.index(tag)
        prev_tag = tag_names[idx + 1] if idx + 1 < len(tag_names) else None
    except ValueError:
        prev_tag = None
    
    if not prev_tag:
        return {"error": "Could not find previous tag", "org": org, "repo": repo, "tag": tag}
    
    # Compare with patches included
    compare_data = await ghe_get(f"/repos/{org}/{repo}/compare/{prev_tag}...{tag}")
    if not compare_data:
        return {"error": "Compare failed", "org": org, "repo": repo}
    
    files = compare_data.get("files", [])
    commits = compare_data.get("commits", [])
    
    return {
        "org": org,
        "repo": repo,
        "tag": tag,
        "previous_tag": prev_tag,
        "total_commits": len(commits),
        "commit_messages": [
            c.get("commit", {}).get("message", "").split("\n")[0]
            for c in commits
        ],
        "files": [
            {
                "filename": f["filename"],
                "status": f.get("status", ""),
                "additions": f.get("additions", 0),
                "deletions": f.get("deletions", 0),
                "patch": f.get("patch", ""),  # Full unified diff for this file
            }
            for f in files
        ],
    }


async def fetch_code_patches_from_commit(org: str, repo: str, commit_sha: str) -> Dict:
    """
    Fetch full patch content for a single commit.
    """
    commit_data = await ghe_get(f"/repos/{org}/{repo}/commits/{commit_sha}")
    if not commit_data:
        return {"error": "Could not fetch commit", "org": org, "repo": repo, "commit": commit_sha}
    
    files = commit_data.get("files", [])
    commit_info = commit_data.get("commit", {})
    
    return {
        "org": org,
        "repo": repo,
        "commit": commit_sha[:12],
        "message": (commit_info.get("message") or ""),
        "author": (commit_info.get("author") or {}).get("name", ""),
        "files": [
            {
                "filename": f["filename"],
                "status": f.get("status", ""),
                "additions": f.get("additions", 0),
                "deletions": f.get("deletions", 0),
                "patch": f.get("patch", ""),
            }
            for f in files
        ],
    }


async def fetch_full_code_changes(deployment: Dict) -> Dict:
    """
    Given a deployment, fetch the actual code patches (line-level diffs).
    Tries changelog URL first, falls back to commit SHA.
    """
    changelog = deployment.get("changelog", "")
    commit = deployment.get("commit", "")
    ghe_org = deployment.get("gheOrg", "")
    ghe_repo = deployment.get("gheRepo", "")
    
    if changelog:
        result = await fetch_code_patches_from_changelog(changelog)
        if "error" not in result:
            return result
    
    if commit and ghe_org and ghe_repo:
        return await fetch_code_patches_from_commit(ghe_org, ghe_repo, commit)
    
    return {"error": "No changelog URL or commit SHA available"}


print("✅ Code patch functions defined.")

✅ Code patch functions defined.


In [226]:
# Fetch actual code patches for a deployment
# This shows the exact lines added/removed — useful for identifying risky code

if results:
    target_dep = results[TARGET_DEPLOYMENT_INDEX]
    print(f"Fetching full code patches for: {target_dep.get('version')}")
    print(f"  Changelog: {target_dep.get('changelog', 'N/A')}")
    print(f"  Commit:    {target_dep.get('commit', '')[:12]}")
    print()

    patches = await fetch_full_code_changes(target_dep)

    if "error" in patches:
        print(f"❌ Error: {patches['error']}")
        print("   Make sure GHE_TOKEN is set (source.datanerd.us/settings/tokens)")
    else:
        print(f"✅ Fetched patches for {len(patches.get('files', []))} files")
        print(f"   Tag: {patches.get('tag', patches.get('commit', ''))}")
        if patches.get("previous_tag"):
            print(f"   Comparing: {patches['previous_tag']} → {patches['tag']}")
        if patches.get("commit_messages"):
            print(f"\n   Commit messages:")
            for msg in patches["commit_messages"][:10]:
                print(f"     • {msg}")

        print(f"\n{'='*80}")
        print("CODE CHANGES (unified diff patches)")
        print(f"{'='*80}")

        for f in patches.get("files", []):
            patch = f.get("patch", "")
            if not patch:
                continue
            print(f"\n{'─'*80}")
            print(f"📄 {f['filename']}  ({f['status']}  +{f['additions']} -{f['deletions']})")
            print(f"{'─'*80}")
            print(patch)

        # Store in deployment for later use (e.g., feeding to LLM)
        target_dep["code_patches"] = patches
        
        # Summary of potentially risky changes
        print(f"\n{'='*80}")
        print("RISK INDICATORS IN CODE")
        print(f"{'='*80}")
        
        risky_patterns = {
            "null/nil handling": ["null", "nil", "None", "undefined", "Optional"],
            "error handling": ["catch", "except", "rescue", "error", "panic", "throw"],
            "concurrency": ["synchronized", "mutex", "lock", "atomic", "goroutine", "async", "await"],
            "database": ["SELECT", "INSERT", "UPDATE", "DELETE", "migration", "schema", "ALTER"],
            "authentication": ["auth", "token", "password", "credential", "session", "jwt"],
            "configuration": ["config", "env", "setting", "property", "flag"],
            "timeout/retry": ["timeout", "retry", "backoff", "deadline", "circuit"],
        }
        
        for f in patches.get("files", []):
            patch = f.get("patch", "")
            if not patch:
                continue
            # Only look at added lines
            added_lines = [line[1:] for line in patch.split("\n") if line.startswith("+") and not line.startswith("+++")]
            
            file_risks = []
            for category, keywords in risky_patterns.items():
                matching_lines = [l for l in added_lines if any(kw.lower() in l.lower() for kw in keywords)]
                if matching_lines:
                    file_risks.append((category, len(matching_lines), matching_lines[:2]))
            
            if file_risks:
                print(f"\n  📄 {f['filename']}:")
                for category, count, examples in file_risks:
                    print(f"    ⚠️  {category} ({count} lines)")
                    for ex in examples:
                        print(f"        +{ex.strip()[:100]}")
else:
    print("No results — run the analysis cell first.")

Fetching full code patches for: pullrequest-3
  Changelog: https://source.datanerd.us/Alerting/condition-standards-auditor/pull/3/files
  Commit:    477ea9b2f92b

✅ Fetched patches for 19 files
   Tag: 

CODE CHANGES (unified diff patches)

────────────────────────────────────────────────────────────────────────────────
📄 .gitignore  (added  +9 -0)
────────────────────────────────────────────────────────────────────────────────
@@ -0,0 +1,9 @@
+# Worktrees
+.worktrees/
+
+# Build outputs
+build/
+.gradle/
+
+# IDE
+.idea/
\ No newline at end of file

────────────────────────────────────────────────────────────────────────────────
📄 build.gradle.kts  (modified  +94 -10)
────────────────────────────────────────────────────────────────────────────────
@@ -8,7 +8,7 @@ plugins {
     id("us.datanerd.coverage") version "8.1.5"
     id("us.datanerd.coverage-report") version "8.1.5"
     id("us.datanerd.docker") version "8.3.0"
-    kotlin("jvm") version "1.9.21"
+    kotlin("jvm") version "2.

## Step 6b — Fetch Error Inbox Data for Deployments with Alert Violations

For each past deployment that triggered alert violations, fetches the actual errors (error class, message, stack trace) that occurred during the violation window. This tells us exactly what went wrong at the code level.

In [223]:
async def fetch_errors_during_violations(
    entity_guid: str,
    account_id: int,
    start_ms: int,
    end_ms: int,
    limit: int = 20,
) -> List[Dict]:
    """
    Fetch TransactionError data during a specific time window.
    
    Returns error groups with class, message, count, and transaction name.
    """
    nrql = (
        f"SELECT count(*) FROM TransactionError "
        f"WHERE entity.guid = '{entity_guid}' "
        f"SINCE {start_ms} UNTIL {end_ms} "
        f"FACET `error.class`, `error.message`, transactionName "
        f"LIMIT {limit}"
    )
    results = await query_nrql(nrql, account_id)
    
    errors = []
    for r in results:
        facet = r.get("facet", [])
        errors.append({
            "error_class": facet[0] if len(facet) > 0 else "",
            "error_message": facet[1] if len(facet) > 1 else "",
            "transaction_name": facet[2] if len(facet) > 2 else "",
            "count": r.get("count", 0),
        })
    return errors


async def fetch_error_samples(
    entity_guid: str,
    account_id: int,
    start_ms: int,
    end_ms: int,
    limit: int = 5,
) -> List[Dict]:
    """
    Fetch individual error samples during a time window.
    """
    nrql = (
        f"SELECT `error.class`, `error.message`, transactionName, "
        f"message, timestamp "
        f"FROM TransactionError "
        f"WHERE entity.guid = '{entity_guid}' "
        f"SINCE {start_ms} UNTIL {end_ms} "
        f"LIMIT {limit}"
    )
    return await query_nrql(nrql, account_id)


async def fetch_error_inbox_for_violated_deployments(
    deployments: List[Dict],
    entity_guid: str,
    account_id: int,
) -> List[Dict]:
    """
    For each deployment that had alert violations, fetch errors that occurred
    between the deployment time and the alert violation open time.
    
    Time window: deploy_timestamp → first violation openedAt
    This captures exactly the errors that appeared AFTER deploy but BEFORE the alert fired.
    """
    enriched = []
    
    for dep in deployments:
        violations = dep.get("root_entity_alert_violations", [])
        if not violations:
            continue
        
        deploy_ts = dep.get("timestamp_ms", 0)
        if not deploy_ts:
            continue
        
        # Find the violation open time
        # violations have openedAt as human-readable string from Step 4
        # We need to re-query to get the raw timestamp, OR parse the string back
        # Let's parse "2025-06-20 10:13:00" back to epoch ms
        violation_end_ms = None
        for v in violations:
            opened_str = v.get("openedAt", "")
            if opened_str:
                try:
                    dt = datetime.strptime(opened_str, "%Y-%m-%d %H:%M:%S")
                    violation_end_ms = int(dt.timestamp() * 1000)
                    break  # Use the first (earliest) violation
                except ValueError:
                    continue
        
        # If we couldn't parse violation time, use deploy + 30 min as fallback
        if not violation_end_ms:
            violation_end_ms = deploy_ts + (30 * 60 * 1000)
        
        # Ensure end is after start
        if violation_end_ms <= deploy_ts:
            violation_end_ms = deploy_ts + (30 * 60 * 1000)
        
        window_minutes = (violation_end_ms - deploy_ts) / 60000
        
        print(f"\n── {dep.get('version', '?')} ({dep.get('timestamp_str', '?')}) ──")
        print(f"  Window: deploy → violation opened ({window_minutes:.1f} min)")
        print(f"  {ms_to_date_string(deploy_ts)} → {ms_to_date_string(violation_end_ms)}")
        
        # Fetch error groups in this window
        error_groups = await fetch_errors_during_violations(
            entity_guid, account_id, deploy_ts, violation_end_ms
        )
        
        # Fetch error samples
        error_samples = await fetch_error_samples(
            entity_guid, account_id, deploy_ts, violation_end_ms
        )
        
        dep["error_inbox"] = {
            "time_window": {
                "start": ms_to_date_string(deploy_ts),
                "end": ms_to_date_string(violation_end_ms),
                "duration_min": round(window_minutes, 1),
            },
            "error_groups": error_groups,
            "error_samples": error_samples,
            "total_error_count": sum(e.get("count", 0) for e in error_groups),
            "unique_error_classes": list(set(e["error_class"] for e in error_groups if e["error_class"])),
        }
        
        # Display
        if error_groups:
            print(f"  Errors found: {dep['error_inbox']['total_error_count']} total")
            print(f"  Unique error classes: {dep['error_inbox']['unique_error_classes']}")
            print(f"  Top errors:")
            for e in sorted(error_groups, key=lambda x: x["count"], reverse=True)[:5]:
                print(f"    [{e['count']:>4}x] {e['error_class']}: {e['error_message'][:80]}")
                if e['transaction_name']:
                    print(f"           in: {e['transaction_name']}")
        else:
            print(f"  No errors found in this window (data may have aged out)")
        
        enriched.append(dep)
    
    return enriched


print("✅ Error inbox functions defined.")
print("   Time window: deployment timestamp → alert violation openedAt")

✅ Error inbox functions defined.
   Time window: deployment timestamp → alert violation openedAt


In [224]:
# Fetch error inbox data for all deployments that had violations

if results:
    # Filter to only deployments with violations
    violated_deploys = [d for d in results if d.get("root_entity_alert_violations")]
    print(f"Deployments with alert violations: {len(violated_deploys)} / {len(results)}")
    print(f"Fetching error inbox for each...\n")
    
    violated_with_errors = await fetch_error_inbox_for_violated_deployments(
        deployments=violated_deploys,
        entity_guid=ENTITY_GUID,
        account_id=ACCOUNT_ID,
    )
    
    # Summary
    print(f"\n{'='*80}")
    print(f"ERROR INBOX SUMMARY")
    print(f"{'='*80}")
    for dep in violated_with_errors:
        inbox = dep.get("error_inbox", {})
        version = dep.get("version", "?")
        total = inbox.get("total_error_count", 0)
        classes = inbox.get("unique_error_classes", [])
        print(f"\n  {version} ({dep.get('timestamp_str', '?')}):")
        print(f"    Errors: {total} total, {len(classes)} unique classes")
        if classes:
            for cls in classes[:5]:
                print(f"      - {cls}")
else:
    print("No results — run the analysis cell first.")

Deployments with alert violations: 17 / 17
Fetching error inbox for each...


── pullrequest-3 (2026-05-13 21:06:03) ──
  Window: deploy → violation opened (0.9 min)
  2026-05-13 21:06:03 → 2026-05-13 21:07:00
NerdGraph: 200 in 1.60s
NerdGraph: 200 in 1.23s
  No errors found in this window (data may have aged out)

── pullrequest-3 (2026-05-13 04:28:30) ──
  Window: deploy → violation opened (0.5 min)
  2026-05-13 04:28:30 → 2026-05-13 04:29:00
NerdGraph: 200 in 1.37s
NerdGraph: 200 in 1.26s
  No errors found in this window (data may have aged out)

── pullrequest-3 (2026-05-13 04:28:30) ──
  Window: deploy → violation opened (0.5 min)
  2026-05-13 04:28:30 → 2026-05-13 04:29:00
NerdGraph: 200 in 1.42s
NerdGraph: 200 in 1.23s
  No errors found in this window (data may have aged out)

── pullrequest-3 (2026-05-13 04:28:29) ──
  Window: deploy → violation opened (0.5 min)
  2026-05-13 04:28:29 → 2026-05-13 04:29:00
NerdGraph: 200 in 1.38s
NerdGraph: 200 in 1.21s
  No errors found in this

CancelledError: 

## Step 6c — Fetch Code Patches for All Historical Deployments with Violations

Fetches the actual code diffs from GHE for every past deployment that triggered alert violations. This is needed for the LLM prompt's `<historical_code_changes_and_diffs>` section.

In [ ]:
# Fetch code patches ONLY for deployments that had alert violations (PARALLEL)
# These are needed for the <historical_code_changes_and_diffs> section in the LLM prompt

async def fetch_patches_for_single_deploy(dep: Dict, semaphore: asyncio.Semaphore) -> None:
    """Fetch code patches for one deployment, with concurrency limit."""
    async with semaphore:
        version = dep.get("version", "?")
        changelog = dep.get("changelog", "")
        commit = dep.get("commit", "")
        ghe_org = dep.get("gheOrg", "")
        ghe_repo = dep.get("gheRepo", "")
        
        # Skip if already fetched
        if dep.get("code_patches") and "error" not in dep.get("code_patches", {}):
            return
        
        # Try changelog URL first
        if changelog:
            patches = await fetch_code_patches_from_changelog(changelog)
            if "error" not in patches:
                dep["code_patches"] = patches
                return
        
        # Fallback: single commit
        if commit and ghe_org and ghe_repo:
            patches = await fetch_code_patches_from_commit(ghe_org, ghe_repo, commit)
            if "error" not in patches:
                dep["code_patches"] = patches
                return
        
        dep["code_patches"] = {"error": "No changelog URL or commit SHA available"}


if results:
    # Strictly filter: only deployments with non-empty alert violations
    violated_deploys = [
        d for d in results 
        if len(d.get("root_entity_alert_violations", [])) > 0
    ]
    
    # Exclude the target deployment (only want historical ones)
    target_id = results[TARGET_DEPLOYMENT_INDEX].get("changeTrackingId")
    historical_violated = [d for d in violated_deploys if d.get("changeTrackingId") != target_id]
    
    print(f"Total deployments: {len(results)}")
    print(f"Deployments WITH alert violations: {len(violated_deploys)}")
    print(f"Historical violated (excluding current): {len(historical_violated)}")
    print(f"GHE_TOKEN set: {'Yes' if GHE_TOKEN else 'No'}")
    
    print(f"\nDeployments to fetch code for:")
    for d in historical_violated:
        n_violations = len(d.get("root_entity_alert_violations", []))
        already = "✓" if d.get("code_patches") and "error" not in d.get("code_patches", {}) else ""
        print(f"  - {d.get('version', '?')} ({d.get('timestamp_str', '?')}) — {n_violations} violations {already}")
    
    if GHE_TOKEN and historical_violated:
        # Filter out already-fetched ones
        to_fetch = [d for d in historical_violated if not (d.get("code_patches") and "error" not in d.get("code_patches", {}))]
        
        if to_fetch:
            print(f"\nFetching code patches for {len(to_fetch)} deployments in parallel (max 5 concurrent)...\n")
            
            semaphore = asyncio.Semaphore(5)
            t0 = time.perf_counter()
            await asyncio.gather(*[fetch_patches_for_single_deploy(d, semaphore) for d in to_fetch])
            elapsed = time.perf_counter() - t0
            
            print(f"\n✅ Done in {elapsed:.1f}s (parallel)")
        else:
            print(f"\nAll already fetched ✓")
        
        # Summary
        fetched = sum(1 for d in historical_violated if d.get("code_patches") and "error" not in d.get("code_patches", {}))
        failed = sum(1 for d in historical_violated if d.get("code_patches") and "error" in d.get("code_patches", {}))
        print(f"\n{'='*60}")
        print(f"Results: {fetched} fetched, {failed} failed, {len(historical_violated)} total")
        print(f"{'='*60}")
        
        for d in historical_violated:
            patches = d.get("code_patches", {})
            if "error" not in patches:
                files = patches.get("files", [])
                total_lines = sum(f.get("additions", 0) + f.get("deletions", 0) for f in files)
                print(f"  ✅ {d.get('version', '?')}: {len(files)} files, {total_lines} lines")
            else:
                print(f"  ❌ {d.get('version', '?')}: {patches.get('error', '?')}")
    elif not GHE_TOKEN:
        print("\n⚠️ GHE_TOKEN not set. Set it in the Step 5 cell and re-run.")
else:
    print("No results — run the analysis cell first.")

## Step 7 — Build LLM Prompt and Get Risk Analysis

Takes all gathered signals (deployment history, alerts, code patches, related entities) and constructs a structured prompt for Claude. The LLM then reasons over the evidence to produce specific red flags and recommendations.

In [179]:
def build_risk_analysis_prompt(
    target_deployment: Dict,
    all_deployments: List[Dict],
    root_entity: Dict = None,
) -> str:
    """
    Builds the structured prompt matching prompt.md format.
    
    Sections:
      1. System prompt (role + 5 dimensions)
      2. <current_deployment_metadata> — service info + related entities
      3. <current_deployment_code_changes> — PR diff
      4. <historical_deployment_outcomes> — past deploys with violations
      5. <historical_code_changes_and_diffs> — code patches from past failed deploys
      6. Analysis instructions (4 sections)
    """
    dep = target_deployment
    
    # Separate history (exclude target)
    history_deployments = [
        d for d in all_deployments 
        if d.get("changeTrackingId") != dep.get("changeTrackingId")
    ]
    
    # Categorize history
    broken_deploys = [d for d in history_deployments if d.get("root_entity_alert_violations") or d.get("root_entity_anomalies")]
    failed_deploys = [d for d in history_deployments if d.get("deployment_result") == "failure"]
    clean_deploys = [d for d in history_deployments if d not in broken_deploys and d not in failed_deploys]
    problem_deploys = broken_deploys + failed_deploys
    
    # Parse timing
    deploy_time = dep.get("timestamp_str", "Unknown")
    day_of_week = ""
    try:
        dt = datetime.fromtimestamp(dep.get("timestamp_ms", 0) / 1000)
        day_of_week = dt.strftime("%A")
    except:
        pass

    # ═══════════════════════════════════════════════════════════════════════════
    # SYSTEM PROMPT
    # ═══════════════════════════════════════════════════════════════════════════
    prompt = """You are an expert Principal Site Reliability Engineer (SRE), Infrastructure Architect, and Risk Analysis AI. Your task is to evaluate an upcoming, pre-deployment software release, compare it systematically against historical deployment data, assign risk scores based on concrete patterns, and provide actionable engineering recommendations.

You must evaluate both the current and historical deployments across these 5 core dimensions:
1. Files changed (Core application logic vs. Non-core configuration/manifests)
2. Code diff size (Micro vs. Moderate vs. Massive scale changes)
3. Code diff semantics (Contextual and operational risk of the actual code changes, dependency bumps, or architectural mutations)
4. Time of deployment (Temporal risks including peak traffic hours, end of week, or operational blackouts)
5. Blast radius (Potential downstream impact on connected services, event streams, or uninstrumented datastores)

Here is the structured context for your analysis:

"""

    # ═══════════════════════════════════════════════════════════════════════════
    # SECTION 1: <current_deployment_metadata>
    # ═══════════════════════════════════════════════════════════════════════════
    prompt += f"""<current_deployment_metadata>
## DEPLOYMENT UNDER REVIEW (PRE-DEPLOY — outcome unknown)
Service: {dep.get('entity_name', 'Unknown')}
Entity GUID: {dep.get('entity_guid', 'Unknown')}
Version: {dep.get('version', 'Unknown')}
Deployer: {dep.get('user', 'Unknown')}
Planned deploy time: {day_of_week} {deploy_time}
Target environment: {dep.get('environment', 'Unknown')}
Team: {dep.get('team', 'Unknown')}
Repo: {dep.get('gheOrg', '')}/{dep.get('gheRepo', '')}
Deploy Mechanism: {dep.get('deployMechanism', 'Unknown')}

NOTE: This deployment has NOT shipped yet. You are evaluating it BEFORE it goes to production. You do NOT know its outcome. Your job is to predict risk based on historical patterns.

## RELATED ENTITIES (potential blast radius)
"""
    related = dep.get("related_entities", [])
    if related:
        prompt += f"{len(related)} connected service(s) that could be affected if this deploy causes issues:\n"
        for rel in related:
            prompt += f"- {rel.get('name', '?')} ({rel.get('domain', '?')}/{rel.get('type', '?')})\n"
    else:
        prompt += "No related entities discovered.\n"
    
    prompt += "</current_deployment_metadata>\n\n\n"

    # ═══════════════════════════════════════════════════════════════════════════
    # SECTION 2: <current_deployment_code_changes>
    # ═══════════════════════════════════════════════════════════════════════════
    prompt += "<current_deployment_code_changes>\n"
    code_patches = dep.get("code_patches", {})
    if code_patches and "error" not in code_patches:
        files = code_patches.get("files", [])
        total_added = sum(f.get("additions", 0) for f in files)
        total_removed = sum(f.get("deletions", 0) for f in files)
        
        prompt += f"""Tag: {code_patches.get('tag', code_patches.get('commit', 'N/A'))}
Previous tag: {code_patches.get('previous_tag', 'N/A')}
Total commits in release: {code_patches.get('total_commits', 'N/A')}
Files changed: {len(files)}
Lines added: +{total_added}, Lines removed: -{total_removed}
"""
        if code_patches.get("commit_messages"):
            prompt += "\n### Commit messages:\n"
            for msg in code_patches["commit_messages"][:10]:
                prompt += f"- {msg}\n"
        
        prompt += "\n### Modified files:\n"
        for f in files:
            prompt += f"- {f['filename']} (+{f['additions']} -{f['deletions']}) [{f['status']}]\n"
        
        prompt += "\n### Code patches:\n"
        patch_chars = 0
        MAX_PATCH_CHARS = 8000
        for f in files:
            patch = f.get("patch", "")
            if not patch:
                continue
            if patch_chars + len(patch) > MAX_PATCH_CHARS:
                prompt += f"\n[... remaining file patches truncated for brevity]\n"
                break
            prompt += f"```diff\n// {f['filename']}\n{patch}\n```\n"
            patch_chars += len(patch)
    else:
        prompt += "Code patches not available (GHE_TOKEN not set or changelog URL not parseable).\n"
    
    prompt += "</current_deployment_code_changes>\n\n\n"

    # ═══════════════════════════════════════════════════════════════════════════
    # SECTION 3: <historical_deployment_outcomes>
    # ═══════════════════════════════════════════════════════════════════════════
    prompt += "<historical_deployment_outcomes>\n"
    prompt += f"""Total previous deploys: {len(history_deployments)}
Infrastructure failures (pods couldn't start): {len(failed_deploys)}
Deploys that caused metric degradation/alerts after shipping: {len(broken_deploys)}
Clean deploys (shipped without any issues): {len(clean_deploys)}
"""

    if problem_deploys:
        prompt += "\n### Previous deploys that CAUSED PROBLEMS:\n"
        for i, d in enumerate(problem_deploys[:8], 1):
            violations = d.get("root_entity_alert_violations", [])
            anomalies = d.get("root_entity_anomalies", {})
            result = d.get("deployment_result", "success")
            
            prompt += f"""
{i}. **{d.get('version', '?')}** ({d.get('timestamp_str', '?')}) — {result.upper()}
   - Deployer: {d.get('user', '?')}
   - Commit: {d.get('commit', '?')[:12]}
   - Environment: {d.get('environment', '?')}
   - Alert violations after deploy: {len(violations)}
"""
            if violations:
                for v in violations[:3]:
                    prompt += f"     - [{v.get('level', '?')}] {v.get('label', '?')} (opened: {v.get('openedAt', '?')})\n"
            if anomalies:
                prompt += f"   - Golden metric anomalies: {list(anomalies.keys())}\n"
            
            # Error inbox data if available
            error_inbox = d.get("error_inbox", {})
            if error_inbox and error_inbox.get("error_groups"):
                prompt += f"   - Errors during violation window ({error_inbox.get('time_window', {}).get('duration_min', '?')} min):\n"
                for e in sorted(error_inbox["error_groups"], key=lambda x: x["count"], reverse=True)[:3]:
                    prompt += f"     - [{e['count']}x] {e['error_class']}: {e['error_message'][:80]}\n"

    if clean_deploys:
        prompt += "\n### Previous deploys that were CLEAN (no issues):\n"
        for d in clean_deploys[:5]:
            prompt += f"- {d.get('version', '?')} ({d.get('timestamp_str', '?')}) by {d.get('user', '?')} — shipped cleanly\n"
    
    prompt += "</historical_deployment_outcomes>\n\n\n"

    # ═══════════════════════════════════════════════════════════════════════════
    # SECTION 4: <historical_code_changes_and_diffs>
    # ═══════════════════════════════════════════════════════════════════════════
    prompt += "<historical_code_changes_and_diffs>\n"
    
    has_historical_patches = False
    for d in problem_deploys[:5]:
        patches = d.get("code_patches", {})
        if not patches or "error" in patches:
            continue
        
        has_historical_patches = True
        files = patches.get("files", [])
        total_added = sum(f.get("additions", 0) for f in files)
        total_removed = sum(f.get("deletions", 0) for f in files)
        
        prompt += f"""
### {d.get('version', '?')} ({d.get('timestamp_str', '?')}) — {d.get('deployment_result', '?').upper()}
Tag: {patches.get('tag', patches.get('commit', 'N/A'))}
Previous tag: {patches.get('previous_tag', 'N/A')}
Commits: {patches.get('total_commits', 'N/A')}
Files changed: {len(files)}, Lines: +{total_added} -{total_removed}
"""
        if patches.get("commit_messages"):
            prompt += "Commit messages:\n"
            for msg in patches["commit_messages"][:5]:
                prompt += f"  - {msg}\n"
        
        prompt += "Modified files:\n"
        for f in files[:10]:
            prompt += f"  - {f['filename']} (+{f['additions']} -{f['deletions']}) [{f['status']}]\n"
        
        # Include patches (limited)
        patch_chars = 0
        MAX_HIST_PATCH_CHARS = 4000
        for f in files:
            patch = f.get("patch", "")
            if not patch:
                continue
            if patch_chars + len(patch) > MAX_HIST_PATCH_CHARS:
                prompt += f"[... remaining patches for {d.get('version', '?')} truncated]\n"
                break
            prompt += f"```diff\n// {f['filename']}\n{patch}\n```\n"
            patch_chars += len(patch)
        prompt += "\n"
    
    if not has_historical_patches:
        prompt += "Historical code diffs not available (GHE_TOKEN not set). Set GHE_TOKEN and re-run Step 5/6 to fetch code patches for past failing deployments.\n"
    
    prompt += "</historical_code_changes_and_diffs>\n\n\n"

    # ═══════════════════════════════════════════════════════════════════════════
    # ANALYSIS INSTRUCTIONS
    # ═══════════════════════════════════════════════════════════════════════════
    current_version = dep.get('version', '?')
    problem_versions = ", ".join(d.get('version', '?') for d in problem_deploys[:5])
    
    prompt += f"""Please process the provided information and generate a comprehensive assessment structured strictly around the following sections:

### 1. Current Deployment Analysis
Analyze the upcoming pre-deployment (`{current_version}`) thoroughly across the following sub-points:
- **Files Changed:** Map whether modifications reside in core business code, configuration schemas, manifest records, or upstream project bill-of-materials (BOM).
- **Code Diff Size:** Explicitly measure the footprint and density of changes (lines added/removed, total files).
- **Semantic / Contextual Analysis:** Deeply assess what the code changes actually execute. Evaluate the risk of the specific libraries, modules, or logic being modified.
- **Time of Deployment:** Evaluate the day-of-week and time-of-day risks relative to normal team operation hours.
- **Blast Radius:** Highlight exactly which microservices, pipelines, event loops, or uninstrumented databases are functionally exposed if this deployment experiences degradation.

### 2. Historical Failure Analysis
Analyze the provided problematic historical releases ({problem_versions}) collectively across the same 5 dimensions. Identify explicit systemic patterns, correlations, or anomalies (e.g., historical dependencies causing failures, specific deployers involved, specific alerts repeatedly firing).

### 3. Dimension Comparison & Risk Matrix
Synthesize your findings into a comprehensive Markdown table comparing the **Current Deployment** dimensions against **Historical Patterns**. Assign an explicit risk score for each category choosing strictly from: **LOW | MEDIUM | HIGH | CRITICAL**.

| Dimension | Current Deployment Details | Historical Pattern Correlation | Risk Rating | Justification & Technical Reasoning |
| :--- | :--- | :--- | :--- | :--- |
| **Files Changed** | | | | |
| **Diff Size** | | | | |
| **Semantic Risk** | | | | |
| **Deployment Time** | | | | |
| **Blast Radius** | | | | |

### 4. Red Flags & Final Recommendations
Provide actionable, high-impact guidance for the engineering and on-call rotation teams:
- **Risk-Prone Lines & Code Points:** Explicitly identify specific files or library version boundaries in the current PR that introduced the primary risk profile.
- **Evidence-Based Flags (Pure History):** Surface critical red flags derived *exclusively* from empirical historical evidence (e.g., current service failure rate trends, identical component alert histories, recurring regression types).
- **Go/No-Go Recommendation:** Provide a definitive structural recommendation (e.g., Proceed, Postpone, Canary-with-Targeted-Tracing, Rollback-Strategy-Verification) detailing specific verification steps necessary before moving code to production.
"""
    return prompt


# Build the prompt for the LATEST deployment (simulating pre-deploy evaluation)
if results:
    target_dep = results[TARGET_DEPLOYMENT_INDEX]
    
    prompt = build_risk_analysis_prompt(
        target_deployment=target_dep,
        all_deployments=results,
    )
    
    print(f"Prompt built: {len(prompt)} characters ({len(prompt)//4} ~tokens)")
    print(f"Simulating PRE-DEPLOY evaluation for: {target_dep.get('version')} → {target_dep.get('environment')}")
    print(f"History used: {len(results) - 1} previous deployments")
    print(f"\n{'='*80}")
    print(prompt)
    print(f"\n... [{len(prompt) - 4000} more characters]")
    print(f"{'='*80}")
else:
    print("No results — run the analysis cell first.")

Prompt built: 13417 characters (3354 ~tokens)
Simulating PRE-DEPLOY evaluation for: release-408 → us-sad-sandwich
History used: 3 previous deployments

You are an expert Principal Site Reliability Engineer (SRE), Infrastructure Architect, and Risk Analysis AI. Your task is to evaluate an upcoming, pre-deployment software release, compare it systematically against historical deployment data, assign risk scores based on concrete patterns, and provide actionable engineering recommendations.

You must evaluate both the current and historical deployments across these 5 core dimensions:
1. Files changed (Core application logic vs. Non-core configuration/manifests)
2. Code diff size (Micro vs. Moderate vs. Massive scale changes)
3. Code diff semantics (Contextual and operational risk of the actual code changes, dependency bumps, or architectural mutations)
4. Time of deployment (Temporal risks including peak traffic hours, end of week, or operational blackouts)
5. Blast radius (Potential down

In [184]:
# Send prompt to Claude via NerdCompletion (internal) or Anthropic API (fallback)

# ── Option 1: NerdCompletion (preferred for NR internal use) ──────────────────
NERD_COMPLETION_TOKEN = os.environ.get("NERD_COMPLETION_TOKEN", "NCT-eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJzY29wZSI6eyJjaGF0X2NvbXBsZXRpb24iOlt7ImNhcGFiaWxpdGllcyI6WyJyZWFkIl0sImVmZmVjdCI6ImFsbG93IiwidGFyZ2V0Ijp7Im1vZGVsIjoiKiIsInByb3ZpZGVyIjoib3BlbmFpIn19XSwiZW1iZWRkaW5ncyI6W3siY2FwYWJpbGl0aWVzIjpbInJlYWQiXSwiZWZmZWN0IjoiYWxsb3ciLCJ0YXJnZXQiOnsibW9kZWwiOiIqIiwicHJvdmlkZXIiOiJvcGVuYWkifX1dLCJwaW5lY29uZSI6W3siY2FwYWJpbGl0aWVzIjpbInJlYWQiXSwiZWZmZWN0IjoiYWxsb3ciLCJ0YXJnZXQiOnsiaW5kZXgiOlsibnJhaS1zdGFnaW5nIiwibG9jYWwtdGVzdHMiXSwibmFtZXNwYWNlIjpbIioiXSwicHJvamVjdCI6Ikdyb2sgVVMifX0seyJjYXBhYmlsaXRpZXMiOlsicmVhZCIsIndyaXRlIiwidXBkYXRlIiwiZGVsZXRlIl0sImVmZmVjdCI6ImFsbG93IiwidGFyZ2V0Ijp7ImluZGV4IjpbInNoYXJlZC1leHBlcmltZW50YWwiXSwibmFtZXNwYWNlIjpbImhhY2thdGhvbi1qdW4yMDI2LTAxIl0sInByb2plY3QiOiJHcm9rIFVTIn19XSwiYWRtaW4iOiJGYWxzZSJ9LCJ2ZXIiOjEsImlhdCI6MTc4MTE4OTU2OS4yNzg0NDMsImp0aSI6ImY2MGJhYTc3LTk5OTAtNDdkNS04MTA4LTQzNzE3OTZmYjEyMyIsImVudiI6InN0YWdpbmciLCJmZWF0dXJlIjoiaGFja2F0aG9uLXBvb2wtMDEiLCJzdWIiOiJoYWNrYXRob24tanVuMjAyNi0wMSIsImF1ZCI6WyJuZXJkLWNvbXBsZXRpb24iXSwiaXNzIjoidGVhbSBhaXIiLCJyZXF1ZXN0ZXIiOiJ0ZWFtIGFpci9Ob25lIn0.e9ALes5-mizhqVh4TNX2w4Mf4xtKtHK-XpbRHEQ5p2E")  # NCT-... token
NERD_COMPLETION_URL = "https://nerd-completion.staging-service.nr-ops.net"

# ── Option 2: Direct Anthropic API (fallback) ────────────────────────────────
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")  # sk-ant-... key


async def call_claude(prompt: str, model: str = "claude-4-5-opus") -> str:
    """
    Call Claude via NerdCompletion (preferred) or direct Anthropic API (fallback).
    
    NerdCompletion:
      - URL: https://nerd-completion.staging-service.nr-ops.net/v1/messages
      - Auth: Bearer NCT-...
      - Models: claude-3-5-sonnet, claude-4-5-sonnet, claude-4-5-opus
    
    Anthropic API:
      - URL: https://api.anthropic.com/v1/messages
      - Auth: x-api-key sk-ant-...
      - Models: claude-sonnet-4-20250514, claude-opus-4-20250514
    """
    
    # Determine which endpoint to use
    if NERD_COMPLETION_TOKEN:
        url = f"{NERD_COMPLETION_URL}/v1/messages"
        headers = {
            "Authorization": f"Bearer {NERD_COMPLETION_TOKEN}",
            "Content-Type": "application/json",
            "anthropic-version": "2023-06-01",
        }
        api_name = "NerdCompletion"
    elif ANTHROPIC_API_KEY:
        url = "https://api.anthropic.com/v1/messages"
        headers = {
            "x-api-key": ANTHROPIC_API_KEY,
            "Content-Type": "application/json",
            "anthropic-version": "2023-06-01",
        }
        # Anthropic API uses different model IDs
        model_map = {
            # "claude-4-5-sonnet": "claude-sonnet-4-20250514",
            "claude-4-5-opus": "claude-opus-4-20250514",
            # "claude-3-5-sonnet": "claude-3-5-sonnet-20241022",
        }
        model = model_map.get(model, model)
        api_name = "Anthropic API"
    else:
        print("❌ No LLM credentials set.")
        print("   Option 1 (recommended): Set NERD_COMPLETION_TOKEN (get from #air-nerd-completion Slack)")
        print("   Option 2: Set ANTHROPIC_API_KEY (from console.anthropic.com)")
        return ""
    
    body = {
        "model": model,
        "max_tokens": 3000,
        "messages": [{"role": "user", "content": prompt}],
    }
    
    async with httpx.AsyncClient() as client:
        print(f"Calling Claude via {api_name} (model: {model})...")
        t0 = time.perf_counter()
        try:
            resp = await client.post(url, headers=headers, json=body, timeout=60)
            elapsed = time.perf_counter() - t0
            resp.raise_for_status()
            data = resp.json()
            text = data["content"][0]["text"]
            print(f"✅ Response received in {elapsed:.1f}s ({len(text)} chars)")
            return text
        except httpx.HTTPStatusError as e:
            print(f"❌ HTTP {e.response.status_code}: {e.response.text[:200]}")
            return ""
        except Exception as e:
            print(f"❌ Error: {e}")
            return ""


def parse_and_display_analysis(response: str):
    """Parse the JSON response and display it nicely."""
    # Extract JSON from response
    text = response.strip()
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0].strip()
    elif "```" in text:
        text = text.split("```")[1].split("```")[0].strip()
    
    try:
        analysis = json.loads(text)
    except json.JSONDecodeError:
        print("⚠️ Could not parse JSON response. Raw output:")
        print(response)
        return response
    
    # Display
    level = analysis.get("risk_level", "UNKNOWN")
    score = analysis.get("risk_score", "?")
    emoji = {"LOW": "✅", "MEDIUM": "⚠️", "HIGH": "🔶", "CRITICAL": "🚨"}.get(level, "❓")
    
    print(f"\n{'='*80}")
    print(f"{emoji}  RISK ANALYSIS: {score}/100 ({level})")
    print(f"{'='*80}")
    print(f"\n{analysis.get('summary', '')}")
    
    red_flags = analysis.get("red_flags", [])
    if red_flags:
        print(f"\n{'─'*80}")
        print(f"RED FLAGS ({len(red_flags)}):")
        print(f"{'─'*80}")
        for i, flag in enumerate(red_flags, 1):
            severity_emoji = {"HIGH": "🚨", "MEDIUM": "⚠️", "LOW": "ℹ️"}.get(flag.get("severity", ""), "•")
            print(f"\n  {i}. {severity_emoji} [{flag.get('severity', '?')}] {flag.get('title', '')}")
            print(f"     Evidence: {flag.get('evidence', '')}")
            print(f"     Action:   {flag.get('recommendation', '')}")
    else:
        print("\n  ✅ No red flags identified — this deployment looks safe.")
    
    print(f"\n{'─'*80}")
    print(f"RECOMMENDATION: {analysis.get('overall_recommendation', '')}")
    
    if analysis.get("similar_past_failures"):
        print(f"\nSIMILAR PAST FAILURES: {analysis.get('similar_past_failures')}")
    print(f"{'─'*80}")
    
    return analysis


# Run the analysis
if prompt:
    response = await call_claude(prompt)
    if response:
        analysis_result = parse_and_display_analysis(response)
    else:
        print("\nNo response. Set one of:")
        print("  NERD_COMPLETION_TOKEN = 'NCT-...'  (ask in #air-nerd-completion Slack)")
        print("  ANTHROPIC_API_KEY = 'sk-ant-...'   (from console.anthropic.com)")
        print("\nAlternatively, copy the prompt from the cell above and paste into claude.ai")
else:
    print("No prompt built — run the previous cell first.")

Calling Claude via NerdCompletion (model: claude-4-5-opus)...
✅ Response received in 57.5s (11375 chars)
⚠️ Could not parse JSON response. Raw output:
# Pre-Deployment Risk Assessment Report
## Service: query-summarizer | Version: release-408 | Environment: production.us-sad-sandwich

---

## 1. Current Deployment Analysis

### Files Changed
| File | Type | Risk Category |
|------|------|---------------|
| `dependency_license_manifest.yml` | License manifest / metadata | Non-core |
| `gradle/libs.versions.toml` | Dependency BOM version pinning | Non-core (indirect runtime impact) |

**Assessment:** Both modified files are configuration/manifest artifacts rather than core application logic. The changes update version references for the `idiomancer-bom` dependency from `11.0.1` → `11.0.2`. No source code files (`src/main/**`, `src/test/**`) are touched. This is a **pure dependency bump** with no direct code changes in this repository.

### Code Diff Size
| Metric | Value | Classification